# NB07 — Documentation Gap Scoring Model

## Objective
We build a composite score (0-100) that estimates each hospital's documentation improvement opportunity. Higher score = larger gap = more potential revenue being left on the table.

**Key Note:** This is NOT a machine learning model — it's a transparent, rules-based scoring system that combines multiple signals of documentation risk in a weighted manner.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Load data
df = pd.read_csv('../../data/outputs/nb06_peer_benchmarks/hospital_with_benchmarks.csv', dtype={'ccn': str})

print(f"Data loaded: {df.shape[0]} hospitals, {df.shape[1]} columns")
print(f"\nFirst few rows:")
print(df.head())
print(f"\nColumn names:")
print(df.columns.tolist())

Data loaded: 3280 hospitals, 69 columns

First few rows:
      ccn                    hospital_name      city state  zip_code  \
0  010001  SOUTHEAST HEALTH MEDICAL CENTER    DOTHAN    AL     36301   
1  010005         MARSHALL MEDICAL CENTERS      BOAZ    AL     35957   
2  010006     NORTH ALABAMA MEDICAL CENTER  FLORENCE    AL     35630   
3  010007         MIZELL MEMORIAL HOSPITAL       OPP    AL     36467   
4  010008      CRENSHAW COMMUNITY HOSPITAL   LUVERNE    AL     36049   

       county   beds bed_size_tier  \
0     HOUSTON  420.0       400-599   
1    MARSHALL  240.0       200-399   
2  LAUDERDALE  338.0       200-399   
3   COVINGTON   99.0         25-99   
4    CRENSHAW   65.0         25-99   

                                     ownership ownership_category  ...  \
0  Government - Hospital District or Authority         Government  ...   
1  Government - Hospital District or Authority         Government  ...   
2                                  Proprietary         For-

## Scoring Components

The composite documentation gap score combines four signals:

1. **CMI Gap Score (40% weight)** — How far below peer CMI is this hospital? A hospital with lower CMI than peers may be under-coding, missing higher-acuity diagnoses or procedures.

2. **Severity Mix Score (30% weight)** — How does the hospital's multi-morbidity case mix (MCC) compare to peers? Low MCC ratio suggests under-capturing of secondary diagnoses.

3. **Payment Efficiency Score (20% weight)** — How does average payment per discharge compare to peers? Lower payment per case (controlling for volume) suggests potential under-coding.

4. **Complexity Adjustment (10% weight)** — Hospital size and DRG diversity. Larger hospitals with diverse case mixes have more opportunities for documentation improvement.

All component scores are normalized to 0-100, where 100 = highest documented gap.
The final score is a weighted average: **0-100**, where >65 = High risk, 50-65 = Medium risk, <50 = Low risk.

In [2]:
# COMPONENT 1: CMI Gap Score (40% weight)
# More negative cmi_z_score = bigger documentation gap
# Cap at ±3 std devs, then rescale so -3 = 100, 0 = 50, +3 = 0

# Flip sign so negative z-score (below peer) becomes positive gap
df['cmi_gap_score'] = np.clip(-df['cmi_z_score'], -3, 3)
# Rescale to 0-100: -3 (biggest gap) -> 100, 0 (at peer) -> 50, +3 (above peer) -> 0
df['cmi_gap_score'] = ((df['cmi_gap_score'] + 3) / 6 * 100).clip(0, 100)

# Handle NaN with neutral score (50)
df['cmi_gap_score'] = df['cmi_gap_score'].fillna(50)

print("CMI Gap Score Distribution:")
print(df['cmi_gap_score'].describe())
print(f"\nMin: {df['cmi_gap_score'].min():.2f}, Max: {df['cmi_gap_score'].max():.2f}")

CMI Gap Score Distribution:
count    3280.000000
mean       50.226067
std        14.907575
min         0.000000
25%        43.189230
50%        50.771144
75%        59.721545
max       100.000000
Name: cmi_gap_score, dtype: float64

Min: 0.00, Max: 100.00


In [3]:
# COMPONENT 2: Severity Mix Score (30% weight)
# Lower MCC ratio relative to peers = bigger gap
# Normalize mcc_gap to 0-100 scale

# mcc_gap is (hospital_mcc - peer_mcc), so negative = gap
# Flip sign: negative gap becomes positive score
df['severity_score'] = np.clip(-df['mcc_gap'], -0.3, 0.3)
# Rescale to 0-100: -0.3 (biggest gap) -> 100, 0 (at peer) -> 50, +0.3 (above peer) -> 0
df['severity_score'] = ((df['severity_score'] + 0.3) / 0.6 * 100).clip(0, 100)

# Handle NaN with neutral score (50)
df['severity_score'] = df['severity_score'].fillna(50)

print("Severity Mix Score Distribution:")
print(df['severity_score'].describe())
print(f"\nMin: {df['severity_score'].min():.2f}, Max: {df['severity_score'].max():.2f}")

Severity Mix Score Distribution:
count    3280.000000
mean       46.376663
std        25.951351
min         0.000000
25%        30.228754
50%        48.962498
75%        58.825834
max       100.000000
Name: severity_score, dtype: float64

Min: 0.00, Max: 100.00


In [4]:
# COMPONENT 3: Payment Efficiency Score (20% weight)
# Lower payment per discharge vs peers indicates potential under-coding
# Normalize payment_gap by its std deviation

payment_gap_std = df['payment_gap'].std()
# Flip sign: negative gap (below peer) becomes positive score
df['payment_score'] = np.clip(-df['payment_gap'] / payment_gap_std, -3, 3)
# Rescale to 0-100: -3 -> 100, 0 -> 50, +3 -> 0
df['payment_score'] = ((df['payment_score'] + 3) / 6 * 100).clip(0, 100)

# Handle NaN with neutral score (50)
df['payment_score'] = df['payment_score'].fillna(50)

print("Payment Efficiency Score Distribution:")
print(df['payment_score'].describe())
print(f"\nMin: {df['payment_score'].min():.2f}, Max: {df['payment_score'].max():.2f}")

Payment Efficiency Score Distribution:
count    3280.000000
mean       50.454758
std        11.988824
min         0.000000
25%        47.530583
50%        51.794049
75%        57.155707
max        94.294102
Name: payment_score, dtype: float64

Min: 0.00, Max: 94.29


In [5]:
# COMPONENT 4: Complexity Adjustment (10% weight)
# Larger, more diverse hospitals have more cases where documentation matters
# Scale drg_diversity percentile and bed count

df['complexity_score'] = (
    df['drg_diversity'].rank(pct=True) * 50 +  # More DRGs = more CDI opportunity
    df['beds'].rank(pct=True) * 50              # More beds = more cases affected
).clip(0, 100)

# Handle NaN with neutral score (50)
df['complexity_score'] = df['complexity_score'].fillna(50)

print("Complexity Adjustment Score Distribution:")
print(df['complexity_score'].describe())
print(f"\nMin: {df['complexity_score'].min():.2f}, Max: {df['complexity_score'].max():.2f}")

Complexity Adjustment Score Distribution:
count    3280.000000
mean       50.439424
std        25.084599
min         0.870071
25%        31.304898
50%        50.000000
75%        69.896285
max        99.982639
Name: complexity_score, dtype: float64

Min: 0.87, Max: 99.98


In [6]:
# COMPOSITE DOCUMENTATION GAP SCORE
# Weighted average of four components

df['doc_gap_score'] = (
    df['cmi_gap_score'] * 0.40 +
    df['severity_score'] * 0.30 +
    df['payment_score'] * 0.20 +
    df['complexity_score'] * 0.10
).round(2)

print("\n" + "="*60)
print("COMPOSITE DOCUMENTATION GAP SCORE DISTRIBUTION")
print("="*60)
print(df['doc_gap_score'].describe())

# Create gap tier categories
df['gap_tier'] = pd.cut(df['doc_gap_score'], 
                         bins=[0, 50, 65, 100], 
                         labels=['Low', 'Medium', 'High'],
                         include_lowest=True)

print(f"\n" + "="*60)
print("DOCUMENTATION RISK TIERS")
print("="*60)
print(df['gap_tier'].value_counts().sort_index())
print(f"\nPercentage distribution:")
print((df['gap_tier'].value_counts(normalize=True).sort_index() * 100).round(1))


COMPOSITE DOCUMENTATION GAP SCORE DISTRIBUTION
count    3280.000000
mean       49.138277
std        11.313182
min        10.770000
25%        40.890000
50%        49.795000
75%        56.440000
max        87.970000
Name: doc_gap_score, dtype: float64

DOCUMENTATION RISK TIERS
gap_tier
Low       1738
Medium    1266
High       276
Name: count, dtype: int64

Percentage distribution:
gap_tier
Low       53.0
Medium    38.6
High       8.4
Name: proportion, dtype: float64


In [7]:
# Top 25 hospitals by documentation gap score

top_25 = df.nlargest(25, 'doc_gap_score')[[
    'ccn', 'hospital_name', 'state', 'beds', 'ownership_category', 
    'cmi', 'peer_cmi_mean', 'doc_gap_score', 'gap_tier'
]].copy()

print("\n" + "="*100)
print("TOP 25 HOSPITALS BY DOCUMENTATION GAP SCORE")
print("="*100)
print(top_25.to_string(index=False))
print("\n" + "="*100)


TOP 25 HOSPITALS BY DOCUMENTATION GAP SCORE
   ccn                                 hospital_name state   beds ownership_category    cmi  peer_cmi_mean  doc_gap_score gap_tier
230275                          HEALTHSOURCE SAGINAW    MI  278.0          Nonprofit 0.8173       1.830367          87.97     High
040018                    BAPTIST HEALTH - VAN BUREN    AR  103.0          Nonprofit 0.8556       1.655114          85.00     High
400134                    THE SAN JORGE HOSPITAL INC    PR  262.0         For-Profit 0.8086       1.808478          85.00     High
180154        PINEVILLE COMMUNITY HEALTH CENTER, INC    KY  120.0          Nonprofit 0.9995       1.627013          83.95     High
260070             PEMISCOT COUNTY MEMORIAL HOSPITAL    MO  167.0          Nonprofit 1.0049       1.627013          83.68     High
360025             FIRELANDS REGIONAL MEDICAL CENTER    OH  400.0          Nonprofit 1.4133       2.005842          83.49     High
400128                  HOSPITAL PAVIA

In [8]:
# Analysis by hospital characteristics

print("\n" + "="*70)
print("ANALYSIS BY HOSPITAL CHARACTERISTICS")
print("="*70)

# By ownership category
print("\n1. Mean Gap Score by Ownership Category:")
print("-" * 70)
ownership_analysis = df.groupby('ownership_category', dropna=False).agg({
    'doc_gap_score': ['mean', 'median', 'std', 'count']
}).round(2)
ownership_analysis.columns = ['Mean', 'Median', 'Std Dev', 'Count']
print(ownership_analysis.sort_values('Mean', ascending=False))

# By bed size tier
print("\n2. Mean Gap Score by Bed Size Tier:")
print("-" * 70)
bed_size_order = ['Small', 'Medium', 'Large']
bed_analysis = df.groupby('bed_size_tier', dropna=False).agg({
    'doc_gap_score': ['mean', 'median', 'std', 'count']
}).round(2)
bed_analysis.columns = ['Mean', 'Median', 'Std Dev', 'Count']
bed_analysis = bed_analysis.reindex([x for x in bed_size_order if x in bed_analysis.index])
print(bed_analysis)

# By teaching intensity
print("\n3. Mean Gap Score by Teaching Status:")
print("-" * 70)
teaching_analysis = df.groupby('is_teaching', dropna=False).agg({
    'doc_gap_score': ['mean', 'median', 'std', 'count']
}).round(2)
teaching_analysis.columns = ['Mean', 'Median', 'Std Dev', 'Count']
teaching_analysis.index = ['Non-Teaching', 'Teaching']
print(teaching_analysis)

# By census region
print("\n4. Mean Gap Score by Census Region:")
print("-" * 70)
region_analysis = df.groupby('census_region', dropna=False).agg({
    'doc_gap_score': ['mean', 'median', 'std', 'count']
}).round(2)
region_analysis.columns = ['Mean', 'Median', 'Std Dev', 'Count']
print(region_analysis.sort_values('Mean', ascending=False))


ANALYSIS BY HOSPITAL CHARACTERISTICS

1. Mean Gap Score by Ownership Category:
----------------------------------------------------------------------
                     Mean  Median  Std Dev  Count
ownership_category                               
Other               49.83   50.99     5.73    234
Nonprofit           49.62   49.47    10.83   1944
Government          48.55   48.33    14.14    451
For-Profit          47.87   47.45    11.93    651

2. Mean Gap Score by Bed Size Tier:
----------------------------------------------------------------------
Empty DataFrame
Columns: [Mean, Median, Std Dev, Count]
Index: []

3. Mean Gap Score by Teaching Status:
----------------------------------------------------------------------
               Mean  Median  Std Dev  Count
Non-Teaching  48.24   48.89    11.49   2113
Teaching      50.76   51.13    10.80   1167

4. Mean Gap Score by Census Region:
----------------------------------------------------------------------
                Mean  Med

In [9]:
# Save output
from pathlib import Path

# Select columns for output
output_cols = [
    'ccn', 'hospital_name', 'state', 'beds', 'bed_size_tier', 'ownership_category',
    'is_teaching', 'census_region', 'is_urban', 'cmi', 'peer_cmi_mean', 'cmi_gap', 'cmi_z_score',
    'mcc_ratio', 'peer_mcc_ratio_mean', 'mcc_gap',
    'avg_payment_per_discharge', 'payment_gap',
    'beds', 'drg_diversity',
    'cmi_gap_score', 'severity_score', 'payment_score', 'complexity_score',
    'doc_gap_score', 'gap_tier'
]

# Keep only columns that exist in dataframe
output_cols = [col for col in output_cols if col in df.columns]

output_df = df[output_cols].copy()

# Create output directory if it doesn't exist
output_dir = Path('../../data/outputs/nb07_gap_scores')
output_dir.mkdir(parents=True, exist_ok=True)

# Save to CSV
output_path = output_dir / 'hospital_gap_scores.csv'
output_df.to_csv(output_path, index=False)

print("\n" + "="*70)
print("OUTPUT SAVED")
print("="*70)
print(f"File: {output_path}")
print(f"Rows: {len(output_df)}")
print(f"Columns: {len(output_df.columns)}")
print(f"\nColumn list:")
for i, col in enumerate(output_df.columns, 1):
    print(f"  {i:2d}. {col}")

print(f"\nSample of output:")
print(output_df.head(10).to_string())


OUTPUT SAVED
File: ../../data/outputs/nb07_gap_scores/hospital_gap_scores.csv
Rows: 3280
Columns: 26

Column list:
   1. ccn
   2. hospital_name
   3. state
   4. beds
   5. bed_size_tier
   6. ownership_category
   7. is_teaching
   8. census_region
   9. is_urban
  10. cmi
  11. peer_cmi_mean
  12. cmi_gap
  13. cmi_z_score
  14. mcc_ratio
  15. peer_mcc_ratio_mean
  16. mcc_gap
  17. avg_payment_per_discharge
  18. payment_gap
  19. beds
  20. drg_diversity
  21. cmi_gap_score
  22. severity_score
  23. payment_score
  24. complexity_score
  25. doc_gap_score
  26. gap_tier

Sample of output:


      ccn                    hospital_name state   beds bed_size_tier ownership_category  is_teaching census_region  is_urban     cmi  peer_cmi_mean   cmi_gap  cmi_z_score  mcc_ratio  peer_mcc_ratio_mean   mcc_gap  avg_payment_per_discharge   payment_gap   beds  drg_diversity  cmi_gap_score  severity_score  payment_score  complexity_score  doc_gap_score gap_tier
0  010001  SOUTHEAST HEALTH MEDICAL CENTER    AL  420.0       400-599         Government         True         South      True  2.0638       2.062833  0.000967     0.003082   0.712575             0.702689  0.009886               14385.230821 -13127.658373  420.0           91.0      49.948641       48.352314      81.657215         81.487366          58.97   Medium
1  010005         MARSHALL MEDICAL CENTERS    AL  240.0       200-399         Government        False         South     False  1.5450       1.726667 -0.181667    -0.845171   0.616415             0.709112 -0.092697                9414.086404  -4461.930660  240.0         